# Análisis FBG: Fibra DW-3 "Suelta" — Sensores + Media Polarizaciones (01/07/2026)

## 🔬 Setup experimental

Este notebook analiza la fibra **DW-3** en una configuración nueva:

- **Fibra "suelta"**: sin tubo PTFE protector y sin peso al final de la fibra.
- **1 solo RTD en cápsula**: RTD-8 (índice 7). Ya no están el RTD-7 y RTD-8 simultáneamente como en medidas anteriores.
- **FBGs**: Se espera que la fibra DW-3 tenga hasta 5 sensores FBG, pero en la medida del 20260202 el 5º FBG (FBG-4 con indexación 0) estaba en una longitud de onda al **límite del rango del interrogador** y no se pudo analizar. Se sospecha que la fibra pueda estar rota y el último FBG no sea visible en el espectro.

### 📋 Objetivos de este análisis:
1. **Investigar el espectro** (rama del tree `peak`): ¿Se ven 5 picos o solo 4?
2. **Comparar cada sensor FBG con el RTD de la cápsula** (RTD-8, índice 7) mediante plots superpuestos de temperatura vs longitud de onda.
3. Identificar qué sensor FBG está más cercano al RTD y correlaciona mejor.

### ⚠️ Notas importantes:
- **Comparación con 20260202**: En aquella fecha, la fibra DW-3 se analizó con tubo PTFE y peso. Solo 3 de los 4 FBGs configurados fueron funcionales (FBG-4 omitido por estar al límite del rango del interrogador).
- **8 RTDs**: 0-3 interiores (1=profundo, 2=medio, 4=superior, 3=superficial), 4-5 vacíos, **7 = cápsula fibra (único)**
- **Acceso a datos**: XRootD remoto vía uproot (no requiere montar /eos/ localmente)

### 💡 Notas sobre el flujo de trabajo:
1. **Visualización de plateaus**: La celda justo antes de la sección 6.2 muestra gráficos superpuestos con líneas verticales marcando los plateaus. Para ver estas líneas correctamente, debes **ejecutar primero la sección 6.3** (definición de plateaus) y **luego re-ejecutar la celda de visualización**.
2. **Evolución temporal completa**: La **Sección 7** al final del notebook muestra todo el rango temporal disponible.

## 1️⃣ Importar Librerías

In [1]:
# Reiniciar kernel si hay problemas con imports
import sys
print(f"Python ejecutable: {sys.executable}")
print(f"Python versión: {sys.version}")

# Verificar que scipy está disponible
try:
    import scipy
    print("✅ scipy disponible")
except ImportError:
    print("❌ scipy no encontrado - instalando...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "scipy"])
    print("✅ scipy instalado - reinicia el kernel")

Python ejecutable: /afs/cern.ch/user/v/vgarciap/rtd-calibration-ana/FBG_Sensitivity_Analysis/.venv-fbg/bin/python
Python versión: 3.11.13 (main, Jun  1 2026, 00:00:00) [GCC 11.5.0 20240719 (Red Hat 11.5.0-14)]
✅ scipy disponible


In [2]:
import numpy as np
import matplotlib.pyplot as plt
import datetime
from scipy import stats
import uproot

print("✅ Librerías importadas correctamente")

✅ Librerías importadas correctamente


## 2️⃣ Cargar Datos del Archivo ROOT

### 🔍 Listar Archivos Disponibles (Opcional)

Si no estás seguro de qué archivos ROOT existen en el directorio, ejecuta esta celda para listarlos.

In [3]:
# 🔧 CONFIGURA AQUÍ LA RUTA DE TU ARCHIVO
# Desde lxplus puedes acceder directamente a /eos/ (montado localmente)
filepath = "/eos/user/j/jcapotor/FBGdata/ROOTFiles/pressure_setup/20260701.root"

# ⚙️ CONFIGURAR TIME_SHIFT (corrección temporal para wavelength)
# NOTA: Ajustar este valor tras inspeccionar los datos
TIME_SHIFT = datetime.timedelta(hours=0)  # ⚠️ A AJUSTAR — inspeccionar desfase

print(f"📂 Cargando archivo ROOT: {filepath}")
print(f"⏱️  TIME_SHIFT configurado: {TIME_SHIFT.total_seconds()/3600:.1f} horas")

# Cargar datos del archivo ROOT
try:
    with uproot.open(filepath) as file:
        # Mostrar trees disponibles
        print(f"✓ Trees disponibles: {file.keys()}")
        
        # Cargar datos de los trees disponibles
        peak_data = file["peak"].arrays(library="np")
        temp_data = file["temp"].arrays(library="np")
        
        # Intentar cargar presión (puede no existir en este dataset)
        try:
            press_data = file["press"].arrays(library="np")
            press_raw = press_data["press"]
            has_press = True
            print("✓ Datos de presión encontrados")
        except Exception:
            has_press = False
            press_data = None
            press_raw = None
            print("⚠️  No hay datos de presión en este dataset")
    
    # 🔍 DIAGNÓSTICO: Ver estructura de peak_data
    print(f"\n🔍 Claves en peak_data: {list(peak_data.keys())}")
    for key in peak_data.keys():
        print(f"   {key}: shape = {peak_data[key].shape}")
    
    # Extraer arrays de datos
    times_peak_raw = peak_data["t"][:, 0]  # Tiempos de wavelength
    times_temp_raw = temp_data["t"]  # Tiempos de temperatura
    
    # IMPORTANTE: Convertir wavelength de metros a nanómetros
    wav_raw = peak_data["wav"] * 1e9  # m -> nm
    
    temp_raw = temp_data["temp"]
    
    # 🔍 Diagnóstico del número de sensores FBG
    n_fbg_sensors = wav_raw.shape[2] if len(wav_raw.shape) == 3 else 1
    print(f"\n🔍 NÚMERO DE SENSORES FBG DETECTADOS: {n_fbg_sensors}")
    print(f"   (Shape wav_raw: {wav_raw.shape})")
    if n_fbg_sensors == 4:
        print("   → Se detectan 4 picos en el espectro")
    elif n_fbg_sensors == 3:
        print("   → Se detectan solo 3 picos — ¿fibra rota? ¿5º FBG no visible?")
    else:
        print(f"   → Se detectan {n_fbg_sensors} picos")
    
    print(f"\n✅ Datos cargados correctamente")
    print(f"   • Wavelength: {wav_raw.shape}")
    print(f"   • Wavelength range: [{np.min(wav_raw[wav_raw > 0]):.2f}, {np.max(wav_raw):.2f}] nm")
    print(f"   • Temperature: {temp_raw.shape}")
    
    # Convertir tiempos a datetime SIN aplicar shift aún
    times_peak_original = np.array([datetime.datetime.utcfromtimestamp(float(t)) for t in times_peak_raw])
    times_temp = np.array([datetime.datetime.utcfromtimestamp(float(t)) for t in times_temp_raw])
    
    # DIAGNÓSTICO: Mostrar rangos SIN SHIFT para comparar
    print(f"\n📅 RANGOS TEMPORALES ORIGINALES (sin shift):")
    print(f"=" * 70)
    print(f"🔵 WAVELENGTH (picos):")
    print(f"   Inicio: {times_peak_original[0].strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"   Fin:    {times_peak_original[-1].strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"\n🔴 TEMPERATURA:")
    print(f"   Inicio: {times_temp[0].strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"   Fin:    {times_temp[-1].strftime('%Y-%m-%d %H:%M:%S')}")
    
    # Calcular diferencia temporal
    diff_inicio = (times_temp[0] - times_peak_original[0]).total_seconds() / 3600
    diff_fin = (times_temp[-1] - times_peak_original[-1]).total_seconds() / 3600
    
    print(f"\n⚠️  DESFASE DETECTADO:")
    print(f"   Temperatura START - Wavelength START: {diff_inicio:+.2f} horas")
    print(f"   Temperatura END - Wavelength END: {diff_fin:+.2f} horas")
    
    if abs(diff_inicio) > 0.5 or abs(diff_fin) > 0.5:
        print(f"\n💡 RECOMENDACIÓN:")
        print(f"   Los datos parecen tener un desfase temporal.")
        if diff_inicio > 0:
            print(f"   Prueba: TIME_SHIFT = datetime.timedelta(hours={abs(diff_inicio):.0f})")
        else:
            print(f"   Prueba: TIME_SHIFT = datetime.timedelta(hours=-{abs(diff_inicio):.0f})")
        print(f"   Luego re-ejecuta esta celda.")
    else:
        print(f"\n✅ Los datos están alineados temporalmente (no necesitas TIME_SHIFT)")
    
    # Aplicar TIME_SHIFT a wavelength
    times_peak = times_peak_original + TIME_SHIFT
    
    # Variable unificada para compatibilidad con el resto del código
    times_raw = times_peak_raw  # Mantener raw para referencia
    times_all = times_peak  # Times con shift aplicado
    
    if TIME_SHIFT.total_seconds() != 0:
        print(f"\n✅ TIME_SHIFT aplicado: {TIME_SHIFT.total_seconds()/3600:.1f}h")
        print(f"   Nuevo rango wavelength:")
        print(f"   Inicio: {times_peak[0].strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"   Fin:    {times_peak[-1].strftime('%Y-%m-%d %H:%M:%S')}")
    
except FileNotFoundError:
    print(f"❌ ERROR: Archivo no encontrado: {filepath}")
    print(f"\n💡 Verifica que el archivo existe en /eos/")
    raise
except Exception as e:
    print(f"❌ ERROR al cargar el archivo: {e}")
    raise

📂 Cargando archivo ROOT: /eos/user/j/jcapotor/FBGdata/ROOTFiles/pressure_setup/20260701.root
⏱️  TIME_SHIFT configurado: 0.0 horas
❌ ERROR: Archivo no encontrado: /eos/user/j/jcapotor/FBGdata/ROOTFiles/pressure_setup/20260701.root

💡 Verifica que el archivo existe en /eos/


FileNotFoundError: [Errno 2] No such file or directory: '/eos/user/j/jcapotor/FBGdata/ROOTFiles/pressure_setup/20260701.root'

### 📊 Visualización rápida de Presión

⚠️ **NOTA**: Los datos de presión pueden no estar disponibles en este dataset. Si no existen, esta celda se saltará automáticamente.

In [ ]:
# Plot simple de presión vs tiempo (si los datos existen)
if has_press:
    fig, ax = plt.subplots(1, 1, figsize=(14, 4))
    
    # Extraer tiempos de presión y datos
    times_press_raw = press_data["t"]
    times_press = np.array([datetime.datetime.utcfromtimestamp(float(t)) for t in times_press_raw])
    
    # Verificar dimensiones de press_raw y seleccionar primer sensor si es necesario
    if len(press_raw.shape) > 1:
        press_data_plot = press_raw[:, 0]  # Usar primer sensor de presión
    else:
        press_data_plot = press_raw
    
    ax.plot(times_press, press_data_plot, linewidth=0.5, alpha=0.8)
    ax.set_ylabel('Presión')
    ax.set_xlabel('Tiempo')
    ax.set_title('Presión vs Tiempo (datos crudos)')
    ax.grid(True, alpha=0.3)
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()
else:
    print("⚠️  No hay datos de presión disponibles en este dataset. Saltando esta celda.")

## 3️⃣ Filtrar Rango Temporal (16:00-17:30)

In [ ]:
# Usar times_all que ya incluye el TIME_SHIFT aplicado
times = times_all  # Ya tiene el shift de 2h aplicado

print(f"📅 Rango COMPLETO de datos disponible (wavelength con TIME_SHIFT aplicado):")
print(f"   Inicio: {times[0].strftime('%Y-%m-%d %H:%M:%S')}")
print(f"   Fin:    {times[-1].strftime('%Y-%m-%d %H:%M:%S')}")

# ============================================================================
# ⬅️ AJUSTA AQUÍ EL RANGO TEMPORAL QUE QUIERES ANALIZAR
# ============================================================================
# Opción A: Usar TODO el rango disponible
use_full_range = False  # Cambiar a True para analizar todos los datos

if use_full_range:
    t_inicio = times[0]
    t_fin = times[-1]
else:
    # Opción B: Definir rango específico manualmente
    # ⚠️ AJUSTA ESTOS VALORES según el rango que viste arriba
    fecha = datetime.date(2026, 2, 2)  # ⬅️ CAMBIAR según tu archivo (20260202.root)
    t_inicio = datetime.datetime.combine(fecha, datetime.time(16, 0))
    t_fin = datetime.datetime.combine(fecha, datetime.time(18, 0))

# Filtrar wavelength por rango temporal (ya tiene shift aplicado)
mask_time = (times >= t_inicio) & (times <= t_fin)
times_filtered = times[mask_time]
wav_filtered = wav_raw[mask_time]

# Filtrar temperatura usando sus propios tiempos (sin shift)
mask_time_temp = (times_temp >= t_inicio) & (times_temp <= t_fin)
temp_filtered = temp_raw[mask_time_temp]
times_temp_filtered = times_temp[mask_time_temp]

print(f"\n✅ Rango seleccionado para análisis:")
print(f"   {t_inicio.strftime('%Y-%m-%d %H:%M')} - {t_fin.strftime('%H:%M')}")
print(f"   Puntos en el rango: {len(times_filtered):,}")

if len(times_filtered) == 0:
    print(f"\n⚠️ ADVERTENCIA: No hay datos en el rango especificado!")
    print(f"\n💡 Soluciones:")
    print(f"   1. Cambia 'use_full_range = True' para usar todos los datos")
    print(f"   2. O ajusta 't_inicio' y 't_fin' según el rango disponible (ver arriba)")

## 4️⃣ Análisis de Temperatura (8 RTDs)

In [ ]:
# Configuración RTDs en orden de altura física (de abajo arriba en la vasija)
rtd_sensors_ordered = [
    (0, "RTD-0 (Interior-Profundo)"),      # Más cercano al suelo
    (1, "RTD-1 (Interior-Medio)"),          # Medio
    (3, "RTD-3 (Interior)"),                # Siguiente altura
    (2, "RTD-2 (Interior-Superficial)"),    # Más externo/superficial
    (4, "RTD-4 (Vacío)"),                   # Vacío
    (5, "RTD-5 (Vacío)"),                   # Vacío
    (6, "RTD-6 (Ext. Cápsula)"),            # Exterior cápsula (puede estar vacío)
    (7, "RTD-7 (Cápsula Fibra)"),           # ✅ ÚNICO RTD en cápsula de fibra
]

n_rtds = len(rtd_sensors_ordered)

# Layout: 2 filas x 4 columnas
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
axes = axes.flatten()

for plot_idx, (rtd_idx, rtd_name) in enumerate(rtd_sensors_ordered):
    ax = axes[plot_idx]
    temp_sensor = temp_filtered[:, rtd_idx]
    mask_valid = temp_sensor > 0
    
    ax.plot(times_temp_filtered[mask_valid], temp_sensor[mask_valid], 
            linewidth=0.5, alpha=0.8)
    ax.set_title(rtd_name, fontsize=10, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis='x', rotation=45)
    
    if mask_valid.sum() > 0:
        ax.set_ylabel('T (K)')
    else:
        ax.text(0.5, 0.5, 'Sin datos', transform=ax.transAxes,
                ha='center', va='center', fontsize=12, color='red')

plt.suptitle('Temperaturas individuales de los 8 RTDs (rango filtrado)',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("\n📌 NOTA: En este setup solo RTD-7 (índice 7) está en la cápsula de la fibra.")
print("   RTD-6 ya NO está en la cápsula (cambio de setup respecto al 20260202).")

## 5️⃣ Procesamiento FBG: Filtrado y Media de Polarizaciones

### 5.1 Configuración de Sensores FBG

⚠️ **NOTA**: La fibra DW-3 puede tener hasta 5 FBGs, pero en el análisis del 20260202 el 5º FBG (FBG-5/índice 4) estaba en una longitud de onda **al límite del rango del interrogador**, por lo que no se pudo analizar. Ahora, con la fibra suelta (sin PTFE ni peso), se sospecha que podría estar rota y el último FBG no sea visible.

**Investigar**: ¿Cuántos picos se ven en el espectro? La celda de carga de datos muestra cuántos sensores detecta el tree `peak`.

In [ ]:
# ⚠️ CONFIGURACIÓN: Detectar sensores automáticamente según los datos cargados
# La fibra DW-3 tiene hasta 5 FBGs, pero el interrogador puede no ver todos
# En 20260202: solo 3 de 4 sensores fueron funcionales (FBG-4 omitido)
# Ahora (fibra suelta, sin PTFE): ¿4 picos? ¿5? ¿Solo 3?

n_fbg_total = wav_raw.shape[2] if len(wav_raw.shape) == 3 else 1
print(f"📊 El archivo ROOT contiene {n_fbg_total} sensores FBG (canales)")

# Detectar sensores con datos válidos (wavelength > 0)
fbg_config = []
active_sensors = []

for sensor_idx in range(n_fbg_total):
    # Verificar si el sensor tiene datos positivos
    wav_sensor_p = wav_raw[:, 0, sensor_idx]  # Polarización P
    wav_sensor_s = wav_raw[:, 1, sensor_idx]  # Polarización S
    
    has_data_p = np.sum(wav_sensor_p > 0) > 100  # Al menos 100 puntos válidos
    has_data_s = np.sum(wav_sensor_s > 0) > 100
    
    sensor_num = sensor_idx + 1  # 1-based para labels
    
    if has_data_p and has_data_s:
        fbg_config.append({"pol": 0, "sensor": sensor_idx, "label": f"FBG-{sensor_num}-P"})
        fbg_config.append({"pol": 1, "sensor": sensor_idx, "label": f"FBG-{sensor_num}-S"})
        active_sensors.append(sensor_num)
        wav_range_p = f"[{np.min(wav_sensor_p[wav_sensor_p>0]):.2f}, {np.max(wav_sensor_p):.2f}]"
        wav_range_s = f"[{np.min(wav_sensor_s[wav_sensor_s>0]):.2f}, {np.max(wav_sensor_s):.2f}]"
        print(f"  ✅ FBG-{sensor_num}: ACTIVO (P: {wav_range_p} nm, S: {wav_range_s} nm)")
    else:
        status_p = '✅' if has_data_p else '❌'
        status_s = '✅' if has_data_s else '❌'
        print(f"  ⚠️  FBG-{sensor_num}: OMITIDO (P: {status_p}, S: {status_s})")

print(f"\n✅ Configurados {len(fbg_config)} canales FBG ({len(active_sensors)} sensores × 2 polarizaciones)")
print(f"   Sensores activos: {active_sensors}")
if len(active_sensors) < n_fbg_total:
    omitted = [i+1 for i in range(n_fbg_total) if (i+1) not in active_sensors]
    print(f"   ⚠️  Sensores omitidos: {omitted}")

### 5.2 Filtrado Wavelength (μ±3σ)

In [ ]:
def filter_wavelength_data(wav_data, timestamps, sigma_threshold=3.0):
    """Filtra wavelength usando método μ±3σ"""
    mask_positive = wav_data > 0
    wav_positive = wav_data[mask_positive]
    
    # Si no hay datos válidos, retornar estructura vacía
    if len(wav_positive) == 0:
        return {
            "timestamps": np.array([]),
            "wavelength": np.array([]),
            "mean": np.nan,
            "std": np.nan,
            "n_valid": 0
        }
    
    mean_wav = np.mean(wav_positive)
    std_wav = np.std(wav_positive)
    threshold_min = mean_wav - sigma_threshold * std_wav
    threshold_max = mean_wav + sigma_threshold * std_wav
    
    mask_filtered = (wav_data > 0) & (wav_data >= threshold_min) & (wav_data <= threshold_max)
    
    return {
        "timestamps": timestamps[mask_filtered],
        "wavelength": wav_data[mask_filtered],
        "mean": mean_wav,
        "std": std_wav,
        "n_valid": np.sum(mask_filtered)
    }

# DIAGNÓSTICO: Verificar forma y valores de los sensores
print(f"🔍 Forma de wav_filtered: {wav_filtered.shape}")
print(f"   Dimensiones: (time={wav_filtered.shape[0]}, pol={wav_filtered.shape[1]}, sensor={wav_filtered.shape[2]})")

# Diagnóstico por sensor (solo sensores funcionales: 0, 1, 2)
print(f"\n🔍 Diagnóstico de valores por sensor (solo FBG-1, FBG-2, FBG-3):")
for sensor_idx in range(3):  # Solo 0, 1, 2 (FBG-4 omitido)
    for pol_idx in range(2):
        pol_name = "P" if pol_idx == 0 else "S"
        sensor_data = wav_filtered[:, pol_idx, sensor_idx]
        n_positive = np.sum(sensor_data > 0)
        if n_positive > 0:
            min_val = np.min(sensor_data[sensor_data > 0])
            max_val = np.max(sensor_data)
            mean_val = np.mean(sensor_data[sensor_data > 0])
        else:
            min_val = max_val = mean_val = 0
        
        print(f"   FBG-{sensor_idx+1}-{pol_name}: {n_positive:7d} pts > 0, range=[{min_val:.2f}, {max_val:.2f}] nm, μ={mean_val:.2f} nm")

# Procesar solo canales configurados (FBG-4 excluido)
fbg_filtered_data = {}
for cfg in fbg_config:
    wav_data = wav_filtered[:, cfg["pol"], cfg["sensor"]]  # (time, pol, sensor)
    fbg_filtered_data[cfg["label"]] = filter_wavelength_data(wav_data, times_filtered)

print("\n✅ Filtrado completado (solo sensores funcionales)")
print(f"\n{'Sensor':<12} {'Mean (nm)':<12} {'Std (nm)':<12} {'N válidos'}")
print("="*60)
for label, data in fbg_filtered_data.items():
    print(f"{label:<12} {data['mean']:<12.6f} {data['std']:<12.6f} {data['n_valid']}")

### 5.3 Calcular Media de Polarizaciones

In [ ]:
# Calcular media P+S para cada sensor activo
fbg_mean_data = {}

for sensor_num in active_sensors:  # Solo sensores detectados como activos
    label_p = f"FBG-{sensor_num}-P"
    label_s = f"FBG-{sensor_num}-S"
    label_mean = f"FBG-{sensor_num}-Mean"
    
    data_p = fbg_filtered_data[label_p]
    data_s = fbg_filtered_data[label_s]
    
    times_p = data_p["timestamps"]
    times_s = data_s["timestamps"]
    wav_p = data_p["wavelength"]
    wav_s = data_s["wavelength"]
    
    # Interpolar para alinear ambas polarizaciones
    # Usar timestamps de P como referencia
    from scipy import interpolate
    
    # Convertir timestamps a numéricos para interpolación
    times_p_num = np.array([t.timestamp() for t in times_p])
    times_s_num = np.array([t.timestamp() for t in times_s])
    
    # Interpolar S sobre los tiempos de P
    f_interp = interpolate.interp1d(times_s_num, wav_s, kind='linear', 
                                     bounds_error=False, fill_value=np.nan)
    wav_s_interp = f_interp(times_p_num)
    
    # Calcular media punto a punto
    wav_mean = (wav_p + wav_s_interp) / 2
    
    # Eliminar NaN
    mask_valid = ~np.isnan(wav_mean)
    
    fbg_mean_data[label_mean] = {
        "timestamps": times_p[mask_valid],
        "wavelength": wav_mean[mask_valid],
        "sensor_num": sensor_num
    }
    
    print(f"✅ {label_mean}: {mask_valid.sum()} puntos válidos, "
          f"rango [{np.nanmin(wav_mean[mask_valid]):.3f}, {np.nanmax(wav_mean[mask_valid]):.3f}] nm")

print(f"\n📊 Media de polarizaciones calculada para {len(fbg_mean_data)} sensores")

### 5.4 Visualizar Wavelengths (Media de Polarizaciones)

In [ ]:
# Ajustar layout para N sensores activos
n_sensors = len(fbg_mean_data)

if n_sensors <= 3:
    fig, axes = plt.subplots(1, n_sensors, figsize=(6*n_sensors, 5))
elif n_sensors == 4:
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
else:
    fig, axes = plt.subplots(1, n_sensors, figsize=(6*n_sensors, 5))

if n_sensors == 1:
    axes = [axes]
else:
    axes = axes.flatten()

for idx, (label, data) in enumerate(fbg_mean_data.items()):
    times_fbg = data["timestamps"]
    wav_fbg = data["wavelength"]
    
    ax = axes[idx]
    ax.plot(times_fbg, wav_fbg, linewidth=0.3, alpha=0.7)
    ax.set_title(label, fontsize=11, fontweight='bold')
    ax.set_ylabel('Wavelength (nm)')
    ax.set_xlabel('Tiempo')
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('Wavelength Media (P+S) por Sensor FBG', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 6️⃣ Correlación Temperatura - Wavelength

### 6.1 Seleccionar RTD de Referencia

En este setup, **solo RTD-7 (índice 7)** está en la cápsula de la fibra.
Comparamos la temperatura de este RTD con cada sensor FBG para determinar cuál correlaciona mejor y está más cercano.

In [ ]:
# Usar RTD-7 (único RTD en cápsula fibra) como referencia
rtd_ref_idx = 7
rtd_ref_name = "RTD-7 (Cápsula)"

temp_ref = temp_filtered[:, rtd_ref_idx]
mask_temp_valid = temp_ref > 0
times_temp_ref = times_temp_filtered[mask_temp_valid]
temp_ref_valid = temp_ref[mask_temp_valid]

print(f"✅ Referencia: {rtd_ref_name}")
print(f"   Rango: [{np.min(temp_ref_valid):.2f}, {np.max(temp_ref_valid):.2f}] K")
print(f"   Puntos válidos: {mask_temp_valid.sum()}")

In [ ]:
# Superposición de temperatura (RTD-7) con cada sensor FBG
# Objetivo: comparar qué sensor FBG correlaciona mejor con el RTD de la cápsula

n_plots = len(fbg_mean_data)
fig, axes = plt.subplots(n_plots, 1, figsize=(16, 5*n_plots), sharex=True)

if n_plots == 1:
    axes = [axes]

color_temp = 'tab:blue'
color_wav = 'tab:red'

for idx, (fbg_label, fbg_data_plot) in enumerate(fbg_mean_data.items()):
    sensor_num = fbg_data_plot['sensor_num']
    ax1 = axes[idx]
    
    # Eje izquierdo: Temperatura
    ax1.set_ylabel(f'{rtd_ref_name} (K)', fontsize=11, color=color_temp)
    ax1.plot(times_temp_ref, temp_ref_valid, color=color_temp, linewidth=1.5, alpha=0.8, label=rtd_ref_name)
    ax1.tick_params(axis='y', labelcolor=color_temp)
    ax1.grid(True, alpha=0.3, axis='both')
    
    # Marcar plateaus con líneas verticales (si están definidos)
    if 'plateau_data' in globals() and len(plateau_data) > 0:
        for plateau in plateau_data:
            ax1.axvline(plateau["t0"], color='red', linestyle='--', linewidth=1.5, alpha=0.6)
            ax1.axvline(plateau["tfin"], color='blue', linestyle='--', linewidth=1.5, alpha=0.6)
    
    # Eje derecho: Wavelength (media punto a punto P+S)
    ax2 = ax1.twinx()
    ax2.set_ylabel('Wavelength (nm)', fontsize=11, color=color_wav)
    ax2.plot(fbg_data_plot["timestamps"], fbg_data_plot["wavelength"],
             color=color_wav, linewidth=1.2, alpha=0.7,
             label=f'FBG-{sensor_num} Mean (P+S)/2')
    ax2.tick_params(axis='y', labelcolor=color_wav)
    
    # Leyendas combinadas
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    
    # Añadir leyenda para plateaus si existen
    if 'plateau_data' in globals() and len(plateau_data) > 0 and idx == 0:
        from matplotlib.lines import Line2D
        custom_lines = [Line2D([0], [0], color='red', linestyle='--', linewidth=1.5),
                        Line2D([0], [0], color='blue', linestyle='--', linewidth=1.5)]
        ax1.legend(lines1 + lines2 + custom_lines,
                   labels1 + labels2 + ['Inicio plateau', 'Fin plateau'],
                   loc='upper left', fontsize=9)
    else:
        ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=9)
    
    # Título del subplot
    ax1.set_title(f'FBG-{sensor_num} vs {rtd_ref_name}: Temperatura vs Wavelength (Media P+S)',
                  fontsize=12, pad=10)

# Solo en el último subplot
axes[-1].set_xlabel('Tiempo', fontsize=12)
for label in axes[-1].get_xticklabels():
    label.set_rotation(45)
    label.set_ha('right')

plt.suptitle(f'Comparación: {rtd_ref_name} vs cada Sensor FBG (Media P+S) — Fibra DW-3 Suelta',
             fontsize=14, y=0.995)
plt.tight_layout()
plt.show()

print(f"\n✅ Gráficos superpuestos generados para {n_plots} sensores FBG vs {rtd_ref_name}")
print(f"   🎯 Objetivo: identificar qué sensor FBG correlaciona mejor con el RTD de la cápsula")
if 'plateau_data' in globals() and len(plateau_data) > 0:
    print(f"   🔴🔵 Plateaus marcados con líneas verticales (rojo=inicio, azul=fin)")
else:
    print(f"   💡 Ejecuta primero la celda 6.3 para ver los plateaus marcados")

In [ ]:
# ======================================================================
# INSPECCIÓN DEL ESPECTRO: ¿Cuántos picos FBG se ven?
# ======================================================================
# En 20260202, la fibra DW-3 tenía 4 canales configurados pero solo 3 funcionales.
# El 5º FBG estaba al límite del rango del interrogador.
# Ahora con la fibra suelta (sin PTFE, sin peso), ¿se ven los mismos picos?

print("="*70)
print("🔍 INSPECCIÓN DEL ESPECTRO FBG")
print("="*70)

# Mostrar rango de wavelength por sensor/canal
n_total_sensors = wav_raw.shape[2] if len(wav_raw.shape) == 3 else 1
print(f"\n📊 Canales totales en el archivo: {n_total_sensors}")
print(f"   Sensores activos detectados: {active_sensors}")

# Para cada sensor, mostrar estadísticas
for i in range(n_total_sensors):
    sensor_num = i + 1
    print(f"\n--- Sensor FBG-{sensor_num} (índice {i}) ---")
    for pol_idx, pol_name in enumerate(['P', 'S']):
        wav_data = wav_raw[:, pol_idx, i]
        mask_pos = wav_data > 0
        n_valid = mask_pos.sum()
        if n_valid > 0:
            print(f"   {pol_name}: {n_valid} puntos válidos, "
                  f"rango [{wav_data[mask_pos].min():.3f}, {wav_data[mask_pos].max():.3f}] nm, "
                  f"media={wav_data[mask_pos].mean():.3f} nm")
        else:
            print(f"   {pol_name}: ❌ SIN DATOS VÁLIDOS")

print(f"\n📌 CONCLUSIÓN:")
if len(active_sensors) == n_total_sensors:
    print(f"   ✅ Todos los {n_total_sensors} sensores tienen datos válidos.")
    if n_total_sensors >= 5:
        print(f"   → Se ven los 5 picos esperados en el espectro.")
    elif n_total_sensors == 4:
        print(f"   → Se ven 4 picos. El 5º FBG sigue sin ser visible.")
    else:
        print(f"   → Se ven solo {n_total_sensors} picos.")
else:
    omitted = [i+1 for i in range(n_total_sensors) if (i+1) not in active_sensors]
    print(f"   ⚠️  Solo {len(active_sensors)} de {n_total_sensors} sensores son funcionales.")
    print(f"   Sensores sin datos: {omitted}")
    print(f"   → Posible rotura de fibra o sensor fuera del rango del interrogador.")

### 6.2 Visualización de Perfiles (para definir plateaus)

Superposición de temperatura y wavelength para identificar visualmente los plateaus.

### 6.3 Definición Manual de Plateaus

**✏️ Define aquí los intervalos temporales de plateau identificados en los gráficos anteriores**

In [ ]:
# ============================================================================
# DEFINICIÓN MANUAL DE PLATEAUS (basados en los gráficos de la sección 6.2)
# ============================================================================
# Formato: [("Nombre", "HH:MM", "HH:MM"), ...]

#manual_plateaus = [
    #("Plateau 1", "16:56", "17:01"),
    #("Plateau 2", "17:06", "17:12"),
    #("Plateau 3", "17:17", "17:21"),
    #("Plateau 4", "17:26", "17:30"),
    #("Plateau 5", "17:40", "17:50"),
#]

manual_plateaus = [
    ("Plateau 1", "16:57", "17:01"),  # 4 min (recortado el inicio)
    ("Plateau 2", "17:08", "17:12"),  # 4 min (recortado el inicio)
    ("Plateau 3", "17:17", "17:21"),  # 4 min (se queda igual)
    ("Plateau 4", "17:26", "17:30"),  # 4 min (se queda igual)
    ("Plateau 5", "17:46", "17:50"),  # 4 min (nos quedamos con los últimos 4 min estables)
]

# Activar plateaus manuales
use_manual_plateaus = True

# Procesar plateaus
plateau_data = []

if use_manual_plateaus and len(manual_plateaus) > 0:
    print("✅ Usando plateaus manuales:")
    print("="*70)
    
    for name, t0_str, tfin_str in manual_plateaus:
        t0 = datetime.datetime.combine(fecha, datetime.datetime.strptime(t0_str, "%H:%M").time())
        tfin = datetime.datetime.combine(fecha, datetime.datetime.strptime(tfin_str, "%H:%M").time())
        
        # Calcular media y std de temperatura en el plateau
        mask_temp_plateau = (times_temp_ref >= t0) & (times_temp_ref <= tfin)
        temp_plateau = temp_ref_valid[mask_temp_plateau]
        
        if len(temp_plateau) > 0:
            temp_mean = np.mean(temp_plateau)
            temp_std = np.std(temp_plateau)
            
            plateau_info = {
                "name": name,
                "t0": t0,
                "tfin": tfin,
                "temp_mean": temp_mean,
                "temp_std": temp_std,
                "n_temp_points": len(temp_plateau)
            }
            
            plateau_data.append(plateau_info)
            
            print(f"{name}: {t0_str} - {tfin_str}")
            print(f"  Temperatura: μ={temp_mean:.3f}K, σ={temp_std:.4f}K, n={len(temp_plateau)}")
        else:
            print(f"⚠️ {name}: No hay datos de temperatura en este rango")
    
    print("="*70)
    print(f"\n✅ {len(plateau_data)} plateaus definidos")
else:
    print("ℹ️ Plateaus desactivados")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("\n" + "="*70)
print("🔬 SINCRONIZACIÓN: BÚSQUEDA DEL LAG ÓPTIMO")
print("="*70)

# Usar RTD-8 como referencia real del análisis
rtd_col = "RTD-8" if "RTD-8" in df_temp_raw.columns else rtd_objetivo

print("""
📋 MÉTODO UTILIZADO:
   1. Barrido exhaustivo: prueba lags entre -300s y +300s
   2. Paso de búsqueda: 1 segundo
   3. Para cada lag: calcula el error MAD (Mean Absolute Deviation)
   4. MAD = promedio de diferencias absolutas entre RTD y wavelength
   5. MAD es más robusto que correlación pura para cambios bruscos
   6. Selecciona el lag que minimiza MAD
   
⚠️  PROBLEMA CON CORRELACIÓN SIMPLE:
   - Correlación maximiza similitud global (engañosa)
   - Con cambios bruscos, puede dar un lag incorrecto
   
✓ VENTAJA DE MAD:
   - Minimiza error punto a punto
   - Detecta desfases grandes correctamente
   - Robusto a cambios abruptos (plateaus, transiciones)
""")

# Interpolamos ambas señales a grid común
t_min = pd.to_datetime("2026-02-02 16:00:00")
t_max = pd.to_datetime("2026-02-02 18:00:00")
time_grid = pd.date_range(start=t_min, end=t_max, freq="1s")

mask_temp = (df_temp_raw["Time_Datetime"] >= t_min) & (df_temp_raw["Time_Datetime"] <= t_max)
temp_times_num_filt = df_temp_raw.loc[mask_temp, "Time_Datetime"].astype(np.int64).values
grid_times_num = time_grid.astype(np.int64).values

df_peak_filt = df_peak_raw[(df_peak_raw["Time_Datetime"] >= t_min) & 
                            (df_peak_raw["Time_Datetime"] <= t_max)].copy()
df_peak_filt = df_peak_filt.sort_values("Time_Datetime").reset_index(drop=True)

if pol_start.upper() == "S":
    df_peak_filt["pol"] = np.where(np.arange(len(df_peak_filt)) % 2 == 0, "S", "P")
else:
    df_peak_filt["pol"] = np.where(np.arange(len(df_peak_filt)) % 2 == 0, "P", "S")

df_peak_filt["pair_id"] = np.arange(len(df_peak_filt)) // 2
df_peak_pairs = df_peak_filt.pivot_table(
    index="pair_id", columns="pol", values=active_wav, aggfunc="first"
)
df_time_pairs = df_peak_filt.groupby("pair_id", as_index=False)["Time_Datetime"].first()
df_peak_mean_16_18 = df_time_pairs.copy()
df_peak_mean_16_18 = df_peak_mean_16_18.merge(df_peak_pairs, left_on="pair_id", right_index=True, how="left")
df_peak_mean_16_18["wav_mean"] = df_peak_mean_16_18[["S", "P"]].mean(axis=1)
df_peak_mean_16_18 = df_peak_mean_16_18.dropna(subset=["wav_mean"])

peak_times_num_mean = df_peak_mean_16_18["Time_Datetime"].astype(np.int64).values

# Interpolar
temp_interp = np.interp(grid_times_num, temp_times_num_filt, df_temp_raw.loc[mask_temp, rtd_col].values)
peak_interp = np.interp(grid_times_num, peak_times_num_mean, df_peak_mean_16_18["wav_mean"].values)

# Normalizar
temp_norm = (temp_interp - np.mean(temp_interp)) / (np.std(temp_interp) + 1e-10)
peak_norm = (peak_interp - np.mean(peak_interp)) / (np.std(peak_interp) + 1e-10)

# Barrido de lags con paso de 1 segundo
lag_range = np.arange(-300, 301, 1)
print(f"\n📊 Evaluando {len(lag_range)} lags con paso de 1 segundo...")

mad_errors = []
for test_lag in lag_range:
    peak_shifted = np.roll(peak_norm, test_lag)
    mad = np.mean(np.abs(temp_norm - peak_shifted))
    mad_errors.append(mad)

# Encontrar óptimo
best_lag_mad_idx = np.argmin(mad_errors)
best_lag_final = lag_range[best_lag_mad_idx]
min_mad = mad_errors[best_lag_mad_idx]

print(f"\n✅ RESULTADO:")
print(f"   Referencia usada: {rtd_col}")
print(f"   Lag óptimo: {best_lag_final}s")
print(f"   Error MAD: {min_mad:.6f}")
print(f"   → FBG llega {abs(best_lag_final)}s {'ANTES' if best_lag_final < 0 else 'DESPUÉS'} que {rtd_col}")
print(f"   → Corrección: sumar {-best_lag_final}s a tiempos del FBG")

# Aplicar corrección
desfase_segundos = best_lag_final
df_peak_mean_16_18["Time_Datetime_Corrected"] = (
    df_peak_mean_16_18["Time_Datetime"] + pd.to_timedelta(desfase_segundos, unit="s")
)

print("\n" + "="*70)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import correlate, correlation_lags

print("\n" + "="*70)
print("🔬 SINCRONIZACIÓN AVANZADA: CORRELACIÓN DE DERIVADAS (FEBRERO)")
print("="*70)

# Aseguramos variables por si se ejecuta sola
if 'rtd_objetivo' not in locals(): rtd_objetivo = "RTD-8"
if 'active_wav' not in locals(): active_wav = "Wav_Ch1"
if 'pol_start' not in locals(): pol_start = "S"

# Usar RTD-8 como referencia real del análisis
rtd_col = "RTD-8" if "RTD-8" in df_temp_raw.columns else rtd_objetivo

print("""
📋 MÉTODO UTILIZADO:
   1. Interpolación de ambas señales a un grid de 1 segundo (16:00 a 18:00).
   2. CÁLCULO DE DERIVADA: Extraemos la velocidad de cambio punto a punto.
   3. Correlación Cruzada: Busca el instante exacto donde las derivadas (caídas) coinciden.
   4. Infalible frente a zonas planas y desfases masivos.
""")

# Interpolamos ambas señales a grid común
t_min = pd.to_datetime("2026-02-02 16:00:00")
t_max = pd.to_datetime("2026-02-02 18:00:00")
time_grid = pd.date_range(start=t_min, end=t_max, freq="1s")

mask_temp = (df_temp_raw["Time_Datetime"] >= t_min) & (df_temp_raw["Time_Datetime"] <= t_max)
temp_times_num_filt = df_temp_raw.loc[mask_temp, "Time_Datetime"].astype(np.int64).values
grid_times_num = time_grid.astype(np.int64).values

df_peak_filt = df_peak_raw[(df_peak_raw["Time_Datetime"] >= t_min) & 
                            (df_peak_raw["Time_Datetime"] <= t_max)].copy()
df_peak_filt = df_peak_filt.sort_values("Time_Datetime").reset_index(drop=True)

if pol_start.upper() == "S":
    df_peak_filt["pol"] = np.where(np.arange(len(df_peak_filt)) % 2 == 0, "S", "P")
else:
    df_peak_filt["pol"] = np.where(np.arange(len(df_peak_filt)) % 2 == 0, "P", "S")

df_peak_filt["pair_id"] = np.arange(len(df_peak_filt)) // 2
df_peak_pairs = df_peak_filt.pivot_table(
    index="pair_id", columns="pol", values=active_wav, aggfunc="first"
)
df_time_pairs = df_peak_filt.groupby("pair_id", as_index=False)["Time_Datetime"].first()
df_peak_mean_16_18 = df_time_pairs.copy()
df_peak_mean_16_18 = df_peak_mean_16_18.merge(df_peak_pairs, left_on="pair_id", right_index=True, how="left")
df_peak_mean_16_18["wav_mean"] = df_peak_mean_16_18[["S", "P"]].mean(axis=1)
df_peak_mean_16_18 = df_peak_mean_16_18.dropna(subset=["wav_mean"])

peak_times_num_mean = df_peak_mean_16_18["Time_Datetime"].astype(np.int64).values

# Interpolar
temp_interp = np.interp(grid_times_num, temp_times_num_filt, df_temp_raw.loc[mask_temp, rtd_col].values)
peak_interp = np.interp(grid_times_num, peak_times_num_mean, df_peak_mean_16_18["wav_mean"].values)

# ====================================================================
# 🚀 MOTOR DE SINCRONIZACIÓN POR DERIVADAS (INFALIBLE)
# ====================================================================
# Calculamos la diferencia (velocidad de cambio) punto a punto
temp_diff = np.diff(temp_interp)
peak_diff = np.diff(peak_interp)

# Normalizamos las derivadas para comparar formas puras
temp_diff_norm = (temp_diff - np.mean(temp_diff)) / (np.std(temp_diff) + 1e-10)
peak_diff_norm = (peak_diff - np.mean(peak_diff)) / (np.std(peak_diff) + 1e-10)

# Aplicamos la correlación cruzada estricta sobre las derivadas
correlation = correlate(temp_diff_norm, peak_diff_norm, mode="full")
lags = correlation_lags(len(temp_diff_norm), len(peak_diff_norm), mode="full")

# Extraemos el lag que maximiza la coincidencia de la caída
best_lag_final = lags[np.argmax(correlation)]

print(f"\n✅ RESULTADO EXACTO:")
print(f"   Referencia usada: {rtd_col}")
print(f"   Lag detectado en el salto: {best_lag_final} segundos")

# Aplicar corrección
desfase_segundos = int(best_lag_final)
df_peak_mean_16_18["Time_Datetime_Corrected"] = (
    df_peak_mean_16_18["Time_Datetime"] + pd.to_timedelta(desfase_segundos, unit="s")
)

print("\n" + "="*70)
print("📊 VISUALIZACIÓN FINAL: ALINEACIÓN A ESCALA REAL")
print("="*70)

# Obtener rango de datos para plot
mask_temp_range = (df_temp_raw["Time_Datetime"] >= t_min) & (df_temp_raw["Time_Datetime"] <= t_max)
df_temp_range = df_temp_raw[mask_temp_range].copy()

# Gráfico con dual-axis
fig_dual, ax1 = plt.subplots(figsize=(16, 6))

color_temp = "tab:blue"
ax1.set_xlabel("Tiempo (Hora Local)", fontsize=12, fontweight="bold")
ax1.set_ylabel(f"{rtd_col} Temperatura (K)", color=color_temp, fontsize=12, fontweight="bold")
line1 = ax1.plot(df_temp_range["Time_Datetime"], df_temp_range[rtd_col], 
                 color=color_temp, linewidth=2.5, alpha=0.8, label=f"{rtd_col}")
ax1.tick_params(axis="y", labelcolor=color_temp)
ax1.grid(True, alpha=0.3, linestyle="--")

ax2 = ax1.twinx()
color_wav = "tab:red"
ax2.set_ylabel(f"{active_wav} MEDIA(S,P) (nm)", color=color_wav, fontsize=12, fontweight="bold")
line2 = ax2.plot(df_peak_mean_16_18["Time_Datetime_Corrected"], df_peak_mean_16_18["wav_mean"],
                 color=color_wav, linewidth=1.5, alpha=0.75, linestyle="--", 
                 label=f"{active_wav} (lag={desfase_segundos}s)")
ax2.tick_params(axis="y", labelcolor=color_wav)

lines = line1 + line2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc="upper right", fontsize=12, framealpha=0.95, edgecolor="black")

ax1.set_title(
    f"✓ Sincronización por Derivadas (Feb) | Lag Aplicado={desfase_segundos}s",
    fontsize=13, fontweight="bold", color="darkgreen", pad=15
)

fig_dual.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# PLOT: Wavelength vs Temperature with Shaded Plateaus (SLIDE OPTIMIZED)
# ============================================================================

import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Configuration
color_temp = 'tab:blue'
color_wav = 'tab:red'
plateau_colors = ['gray', 'green', 'orange', 'purple', 'cyan']

# Aumentamos el tamaño de la figura para que tenga más resolución
fig, axes = plt.subplots(3, 1, figsize=(18, 16), sharex=True)

for idx, sensor_num in enumerate([1, 2, 3]):
    fbg_label = f"FBG-{sensor_num}-Mean"
    if fbg_label not in fbg_mean_data:
        continue
        
    fbg_data_plot = fbg_mean_data[fbg_label]
    ax1 = axes[idx]
    
    # 1. Left Axis: Temperature
    # Aumentamos fontsize a 18 y ponemos negrita extra
    ax1.set_ylabel(f'Temp (K)', fontsize=18, color=color_temp, fontweight='black')
    ax1.plot(times_temp_ref, temp_ref_valid, color=color_temp, linewidth=2.5, alpha=0.9, label='Temperature (RTD-6)')
    
    # Ticks más grandes y en negrita
    ax1.tick_params(axis='y', labelcolor=color_temp, labelsize=15)
    for label in ax1.get_yticklabels():
        label.set_fontweight('bold')
        
    ax1.grid(True, alpha=0.3, linestyle='--', linewidth=1.5)
    
    # 2. Add Shaded Plateaus
    if 'plateau_data' in globals() and len(plateau_data) > 0:
        for p_idx, plateau in enumerate(plateau_data):
            p_color = plateau_colors[p_idx % len(plateau_colors)]
            ax1.axvspan(plateau["t0"], plateau["tfin"], 
                        color=p_color, alpha=0.2, 
                        label=f'Plateau {p_idx+1}' if idx == 0 else "")
            
            if idx == 0:
                mid_time = plateau["t0"] + (plateau["tfin"] - plateau["t0"]) / 2
                # Etiquetas de plateau más grandes (P1, P2...)
                ax1.text(mid_time, ax1.get_ylim()[1], f'P{p_idx+1}', 
                         ha='center', va='bottom', fontsize=16, color='black', fontweight='bold')

    # 3. Right Axis: Wavelength
    ax2 = ax1.twinx()
    ax2.set_ylabel('Wavelength (nm)', fontsize=18, color=color_wav, fontweight='black')
    # Línea de wavelength más gruesa para visibilidad
    ax2.plot(fbg_data_plot["timestamps"], fbg_data_plot["wavelength"], 
             color=color_wav, linewidth=2.0, alpha=0.8, 
             label=f'FBG-{sensor_num} (Mean)')
    
    ax2.tick_params(axis='y', labelcolor=color_wav, labelsize=15)
    for label in ax2.get_yticklabels():
        label.set_fontweight('bold')
    
    # Titles de cada subplot más grandes
    ax1.set_title(f'FBG Sensor {sensor_num}: Temperature vs. Wavelength Analysis', 
                  fontsize=20, pad=20, fontweight='bold', color='black')
    
    # Leyendas más legibles y con sombra para destacar
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    leg = ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', 
                     frameon=True, framealpha=0.9, fontsize=14, shadow=True)
    plt.setp(leg.get_texts(), fontweight='bold')

# Format X-Axis (Time)
axes[-1].set_xlabel('Time (HH:MM)', fontsize=20, fontweight='black', labelpad=15)
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))

# Ticks del eje X más grandes
for label in axes[-1].get_xticklabels():
    label.set_rotation(0) # He puesto 0 en vez de 45 para que sea más fácil de leer en slide
    label.set_ha('center')
    label.set_fontsize(16)
    label.set_fontweight('bold')

# Título Global GIGANTE para la slide
plt.suptitle('DW-3 (Ormocer-T): Run1 Analysis - FBG Wavelength & RTD Temp Profiles', 
             fontsize=26, y=1.03, fontweight='black', color='darkblue')

plt.tight_layout()
plt.show()

print("✅ Slide-ready plot generated with enhanced visibility.")

In [ ]:
# ============================================================================
# PLOT: Dual Polarization (P & S) vs Temp (SLIDE OPTIMIZED & FIXED AXIS OFFSET)
# Fiber: DW-3 (Ormocer-T) | Run 1 | 2026-02-02
# ============================================================================

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as ticker

# Configuration
color_temp = 'tab:blue'
color_pol_p = 'firebrick'   
color_pol_s = 'royalblue'   
plateau_colors = ['gray', 'green', 'orange', 'purple', 'cyan']

fig, axes = plt.subplots(3, 1, figsize=(18, 16), sharex=True)

for idx, sensor_num in enumerate([1, 2, 3]):
    label_p = f"FBG-{sensor_num}-P"
    label_s = f"FBG-{sensor_num}-S"
    
    if label_p not in fbg_filtered_data or label_s not in fbg_filtered_data:
        continue
        
    data_p = fbg_filtered_data[label_p]
    data_s = fbg_filtered_data[label_s]
    ax1 = axes[idx]
    
    # 1. Left Axis: Temperature
    ax1.set_ylabel(f'Temp (K)', fontsize=18, color=color_temp, fontweight='black')
    ax1.plot(times_temp_ref, temp_ref_valid, color=color_temp, linewidth=2.5, alpha=0.5, label='Temperature')
    ax1.tick_params(axis='y', labelcolor=color_temp, labelsize=15)
    for label in ax1.get_yticklabels():
        label.set_fontweight('bold')
    ax1.grid(True, alpha=0.3, linestyle='--', linewidth=1.5)
    
    # 2. Add Shaded Plateaus
    if 'plateau_data' in globals() and len(plateau_data) > 0:
        for p_idx, plateau in enumerate(plateau_data):
            p_color = plateau_colors[p_idx % len(plateau_colors)]
            ax1.axvspan(plateau["t0"], plateau["tfin"], color=p_color, alpha=0.2)
            if idx == 0:
                mid_time = plateau["t0"] + (plateau["tfin"] - plateau["t0"]) / 2
                ax1.text(mid_time, ax1.get_ylim()[1], f'P{p_idx+1}', 
                         ha='center', va='bottom', fontsize=16, fontweight='bold', color='black')

    # 3. Right Axis: Wavelength (P & S)
    ax2 = ax1.twinx()
    ax2.set_ylabel('Wavelength (nm)', fontsize=18, color='black', fontweight='black')
    
    ax2.plot(data_p["timestamps"], data_p["wavelength"], 
             color=color_pol_p, linewidth=2.0, alpha=0.8, label=f'FBG-{sensor_num}-P')
    ax2.plot(data_s["timestamps"], data_s["wavelength"], 
             color=color_pol_s, linewidth=2.0, alpha=0.8, label=f'FBG-{sensor_num}-S')
    
    ax2.tick_params(axis='y', labelcolor='black', labelsize=15)
    
    # --- FIX: AJUSTE DEL FACTOR COMÚN (OFFSET/SCIENTIFIC NOTATION) ---
    # Esto asegura que el "1.55" o el "+1e3" se vea grande y en negrita
    y_formatter = ax2.yaxis.get_major_formatter()
    ax2.yaxis.set_major_formatter(ticker.ScalarFormatter(useOffset=True))
    ax2.yaxis.get_offset_text().set_fontsize(16)
    ax2.yaxis.get_offset_text().set_fontweight('bold')
    
    for label in ax2.get_yticklabels():
        label.set_fontweight('bold')
    
    # Estilo de Subplots
    ax1.set_title(f'FBG Sensor {sensor_num}: Dual Polarization (P & S)', 
                  fontsize=20, pad=20, fontweight='bold')
    
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    leg = ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', 
                     frameon=True, framealpha=0.9, fontsize=14, shadow=True)
    plt.setp(leg.get_texts(), fontweight='bold')

# Format X-Axis
axes[-1].set_xlabel('Time (HH:MM UTC)', fontsize=20, fontweight='black', labelpad=15)
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))

for label in axes[-1].get_xticklabels():
    label.set_fontsize(16)
    label.set_fontweight('bold')

# Global Title
plt.suptitle('DW-3 (Ormocer-T): Run 1 - Wavelength (Dual Pol) & RTD Temp Profiles', 
             fontsize=26, y=0.98, fontweight='black', color='darkblue')

plt.tight_layout(rect=[0, 0.03, 1, 0.94])
plt.show()

### 6.4 Calcular Sensibilidad por Plateaus

Calcula la media y std de wavelength en cada plateau para cada sensor FBG.

In [ ]:
# Calcular media y std de wavelength por plateau para cada sensor
plateau_sensitivity_data = {}

for fbg_label, fbg_data in fbg_mean_data.items():
    sensor_num = fbg_data["sensor_num"]
    fbg_times = fbg_data["timestamps"]
    fbg_wav = fbg_data["wavelength"]
    
    plateau_wav_stats = []
    
    for plateau in plateau_data:
        # Máscara para datos de wavelength en este plateau
        mask_wav_plateau = (fbg_times >= plateau["t0"]) & (fbg_times <= plateau["tfin"])
        wav_plateau = fbg_wav[mask_wav_plateau]
        
        if len(wav_plateau) > 0:
            wav_mean = np.mean(wav_plateau)
            wav_std = np.std(wav_plateau)
            
            plateau_wav_stats.append({
                "name": plateau["name"],
                "temp_mean": plateau["temp_mean"],
                "temp_std": plateau["temp_std"],
                "wav_mean": wav_mean,
                "wav_std": wav_std,
                "n_wav_points": len(wav_plateau)
            })
    
    plateau_sensitivity_data[fbg_label] = plateau_wav_stats

print("✅ Estadísticas de wavelength calculadas por plateau")
print("\nResumen por sensor:")
print("="*80)
for fbg_label in fbg_mean_data.keys():
    sensor_num = fbg_mean_data[fbg_label]["sensor_num"]
    n_plateaus = len(plateau_sensitivity_data[fbg_label])
    print(f"FBG-{sensor_num}: {n_plateaus} plateaus procesados")

### 6.5 Visualizar Sensibilidad (Temp vs Wavelength con barras de error)

In [ ]:
# ============================================================================
# FINAL SENSITIVITY ANALYSIS: Fiber DW-3 | Run 1 | SINGLE HIGH-VISIBILITY FIGURE
# ============================================================================

import matplotlib.pyplot as plt
from scipy import stats
import numpy as np
import matplotlib.ticker as ticker

# Estilo global para máxima visibilidad en proyector
plt.rcParams.update({'font.weight': 'bold', 'axes.labelweight': 'bold'})

# Estructura para guardar resultados
summary_results = []
fiber_info = "Fiber: DW-3 (Ormocer-T) | Run 1 | 2026-02-02"

# --- CREAMOS UNA SOLA FIGURA (3 Sensores x 3 Tipos = 9 Subplots o 3x3) ---
# He optado por una matriz 3x3 para que cada sensor tenga su fila
fig, axes = plt.subplots(3, 3, figsize=(24, 20))

for s_idx, sensor_num in enumerate([1, 2, 3]):
    plot_configs = [
        {"key": f"FBG-{sensor_num}-P", "title": "Pol P", "color": "firebrick"},
        {"key": f"FBG-{sensor_num}-S", "title": "Pol S", "color": "royalblue"},
        {"key": f"FBG-{sensor_num}-Mean", "title": "Mean (P+S)/2", "color": "forestgreen"}
    ]

    for p_idx, config in enumerate(plot_configs):
        ax = axes[s_idx, p_idx]
        label = config["key"]
        
        t_means, t_stds, w_means, w_stds = [], [], [], []
        
        if 'plateau_data' in globals() and len(plateau_data) > 0:
            for p in plateau_data:
                t_means.append(p["temp_mean"])
                t_stds.append(p["temp_std"])
                source = fbg_mean_data if "Mean" in label else fbg_filtered_data
                
                if label in source:
                    mask = (source[label]["timestamps"] >= p["t0"]) & \
                           (source[label]["timestamps"] <= p["tfin"])
                    wav_data = source[label]["wavelength"][mask]
                    if len(wav_data) > 0:
                        w_means.append(np.mean(wav_data)); w_stds.append(np.std(wav_data))
                    else:
                        w_means.append(np.nan); w_stds.append(np.nan)

            x = np.array(t_means); y = np.array(w_means)
            x_err = np.array(t_stds); y_err = np.array(w_stds)
            mask_valid = ~np.isnan(x) & ~np.isnan(y)
            x, y, x_err, y_err = x[mask_valid], y[mask_valid], x_err[mask_valid], y_err[mask_valid]

            if len(x) > 1:
                slope, intercept, r_val, p_val, std_err = stats.linregress(x, y)
                slope_pm, err_pm = slope * 1e3, std_err * 1e3
                
                summary_results.append({
                    "Sensor": f"S{sensor_num}", "Type": config["title"],
                    "Slope": slope_pm, "Error": err_pm, "R2": r_val**2
                })

                # --- PLOT DE ALTA VISIBILIDAD ---
                # Puntos y barras de error más gruesas
                ax.errorbar(x, y, xerr=x_err, yerr=y_err, fmt='o', color=config["color"], 
                            ecolor='black', elinewidth=2.5, capsize=6, markersize=5,
                            label='Experimental Data')
                
                # Línea de ajuste más marcada
                ax.plot(x, slope*x + intercept, color='black', linestyle='--', linewidth=3, alpha=0.7)

                # --- LEYENDA GIGANTE CON LA SLOPE ---
                slope_label = f"SLOPE: {slope_pm:.2f} ± {err_pm:.2f} pm/K\n$R^2$: {r_val**2:.4f}"
                leg = ax.legend([slope_label], loc='upper left', fontsize=18, 
                                 frameon=True, shadow=True, facecolor='white', edgecolor='black')
                plt.setp(leg.get_texts(), fontweight='black')

                # Estilo de ejes (Grande y Negrita)
                ax.set_title(f"SENSOR {sensor_num} | {config['title']}", fontsize=22, fontweight='black', pad=15)
                ax.set_xlabel("Temperature (K)", fontsize=18, fontweight='bold')
                ax.set_ylabel("Wavelength (nm)", fontsize=18, fontweight='bold')
                
                # Números de los ejes grandes
                ax.tick_params(axis='both', which='major', labelsize=16)
                for label_tick in ax.get_xticklabels() + ax.get_yticklabels():
                    label_tick.set_fontweight('bold')

                # Factor común (Offset) grande
                ax.yaxis.set_major_formatter(ticker.ScalarFormatter(useOffset=True))
                ax.yaxis.get_offset_text().set_fontsize(16)
                ax.yaxis.get_offset_text().set_fontweight('bold')
                
                ax.grid(True, alpha=0.4, linestyle='--', linewidth=1.5)

# Título Global para la Slide
plt.suptitle(f"SENSITIVITY ANALYSIS SUMMARY: {fiber_info}", 
             fontsize=32, y=1.02, fontweight='black', color='darkblue')

plt.tight_layout()
plt.show()

# ============================================================================
# FINAL SUMMARY TABLE (IMPRESIÓN EN CONSOLA)
# ============================================================================
print("\n" + "="*95)
print(f"{'SENSITIVITY SUMMARY TABLE: FIBER DW-3 (Run 1)':^95}")
print("="*95)
print(f"{'Sensor':<8} | {'Type':<15} | {'Slope (pm/K)':<18} | {'Error (pm/K)':<15} | {'R²':<8}")
print("-" * 95)
for res in summary_results:
    print(f"{res['Sensor']:<8} | {res['Type']:<15} | {res['Slope']:>12.3f}       | {res['Error']:>12.3f}    | {res['R2']:.4f}")
print("="*95)

In [ ]:
# ============================================================================
# FINAL SENSITIVITY ANALYSIS: Fiber DW-3 | Run 1 | SINGLE HIGH-VISIBILITY FIGURE
# ============================================================================

import matplotlib.pyplot as plt
from scipy import stats
import numpy as np
import matplotlib.ticker as ticker

# Estilo global para máxima visibilidad en proyector
plt.rcParams.update({'font.weight': 'bold', 'axes.labelweight': 'bold'})

summary_results = []
fiber_info = "Fiber: DW-3 (Ormocer-T) | Run 1 | 2026-02-02"

fig, axes = plt.subplots(3, 3, figsize=(24, 20))

for s_idx, sensor_num in enumerate([1, 2, 3]):
    plot_configs = [
        {"key": f"FBG-{sensor_num}-P", "title": "Pol P", "color": "firebrick"},
        {"key": f"FBG-{sensor_num}-S", "title": "Pol S", "color": "royalblue"},
        {"key": f"FBG-{sensor_num}-Mean", "title": "Mean (P+S)/2", "color": "forestgreen"}
    ]

    for p_idx, config in enumerate(plot_configs):
        ax = axes[s_idx, p_idx]
        label = config["key"]
        
        # Aquí guardaremos las medias y sus correspondientes SEM (Error de la media)
        t_means, t_sems, w_means, w_sems = [], [], [], []
        
        if 'plateau_data' in globals() and len(plateau_data) > 0:
            for p in plateau_data:
                source = fbg_mean_data if "Mean" in label else fbg_filtered_data
                
                if label in source:
                    mask = (source[label]["timestamps"] >= p["t0"]) & \
                           (source[label]["timestamps"] <= p["tfin"])
                    wav_data = source[label]["wavelength"][mask]
                    
                    if len(wav_data) > 0:
                        # Número de muestras en este plateau
                        n_samples = len(wav_data)
                        
                        # Guardamos las medias de X e Y
                        t_means.append(p["temp_mean"])
                        w_means.append(np.mean(wav_data))
                        
                        # --- NUEVO: CÁLCULO DEL ERROR DE LA MEDIA (SEM) ---
                        # Para longitud de onda usamos stats.sem de Scipy
                        w_sems.append(stats.sem(wav_data))
                        
                        # Para temperatura, calculamos el SEM usando la desviación estándar y la raíz de N
                        # (Asumiendo que el sensor de temperatura y el FBG miden en ventanas temporales equivalentes)
                        t_sems.append(p["temp_std"] / np.sqrt(n_samples))
                    else:
                        w_means.append(np.nan); w_sems.append(np.nan)
                        t_means.append(np.nan); t_sems.append(np.nan)

            x = np.array(t_means); y = np.array(w_means)
            x_err = np.array(t_sems); y_err = np.array(w_sems)  # <--- Ahora usamos los SEM
            mask_valid = ~np.isnan(x) & ~np.isnan(y)
            x, y, x_err, y_err = x[mask_valid], y[mask_valid], x_err[mask_valid], y_err[mask_valid]

            if len(x) > 1:
                slope, intercept, r_val, p_val, std_err = stats.linregress(x, y)
                slope_pm, err_pm = slope * 1e3, std_err * 1e3
                
                summary_results.append({
                    "Sensor": f"S{sensor_num}", "Type": config["title"],
                    "Slope": slope_pm, "Error": err_pm, "R2": r_val**2
                })

                # --- PLOT DE ALTA VISIBILIDAD (Con barras de error SEM) ---
                ax.errorbar(x, y, xerr=x_err, yerr=y_err, fmt='o', color=config["color"], 
                            ecolor='black', elinewidth=2.5, capsize=6, markersize=5,
                            label='Experimental Data')
                
                ax.plot(x, slope*x + intercept, color='black', linestyle='--', linewidth=3, alpha=0.7)

                # --- LEYENDA GIGANTE CON LA SLOPE ---
                slope_label = f"SLOPE: {slope_pm:.2f} $\pm$ {err_pm:.2f} pm/K\n$R^2$: {r_val**2:.4f}"
                leg = ax.legend([slope_label], loc='upper left', fontsize=18, 
                                 frameon=True, shadow=True, facecolor='white', edgecolor='black')
                plt.setp(leg.get_texts(), fontweight='black')

                ax.set_title(f"SENSOR {sensor_num} | {config['title']}", fontsize=22, fontweight='black', pad=15)
                ax.set_xlabel("Temperature (K)", fontsize=18, fontweight='bold')
                ax.set_ylabel("Wavelength (nm)", fontsize=18, fontweight='bold')
                
                ax.tick_params(axis='both', which='major', labelsize=16)
                for label_tick in ax.get_xticklabels() + ax.get_yticklabels():
                    label_tick.set_fontweight('bold')

                ax.yaxis.set_major_formatter(ticker.ScalarFormatter(useOffset=True))
                ax.yaxis.get_offset_text().set_fontsize(16)
                ax.yaxis.get_offset_text().set_fontweight('bold')
                
                ax.grid(True, alpha=0.4, linestyle='--', linewidth=1.5)

plt.suptitle(f"SENSITIVITY ANALYSIS SUMMARY: {fiber_info}", 
             fontsize=32, y=1.02, fontweight='black', color='darkblue')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# FINAL SENSITIVITY ANALYSIS (SEM): Fiber DW-3 | Run 1 | SINGLE HIGH-VISIBILITY FIGURE
# ============================================================================

import matplotlib.pyplot as plt
from scipy import stats
import numpy as np
import matplotlib.ticker as ticker

# Estilo global para máxima visibilidad en proyector
plt.rcParams.update({'font.weight': 'bold', 'axes.labelweight': 'bold'})

# Estructura para guardar resultados
summary_results = []
fiber_info = "Fiber: DW-3 (Ormocer-T) | Run 1 | 2026-02-02"

# --- CREAMOS UNA SOLA FIGURA (3 Sensores x 3 Tipos = 9 Subplots o 3x3) ---
fig, axes = plt.subplots(3, 3, figsize=(24, 20))

for s_idx, sensor_num in enumerate([1, 2, 3]):
    plot_configs = [
        {"key": f"FBG-{sensor_num}-P", "title": "Pol P", "color": "firebrick"},
        {"key": f"FBG-{sensor_num}-S", "title": "Pol S", "color": "royalblue"},
        {"key": f"FBG-{sensor_num}-Mean", "title": "Mean (P+S)/2", "color": "forestgreen"}
    ]

    for p_idx, config in enumerate(plot_configs):
        ax = axes[s_idx, p_idx]
        label = config["key"]
        
        t_means, t_sems, w_means, w_sems = [], [], [], []
        
        if 'plateau_data' in globals() and len(plateau_data) > 0:
            for p in plateau_data:
                source = fbg_mean_data if "Mean" in label else fbg_filtered_data
                
                if label in source:
                    mask = (source[label]["timestamps"] >= p["t0"]) & \
                           (source[label]["timestamps"] <= p["tfin"])
                    wav_data = source[label]["wavelength"][mask]

                    if len(wav_data) > 1:
                        # --- Temperature SEM ---
                        t_mean = p["temp_mean"]
                        t_sem = p["temp_std"] / np.sqrt(max(1, p.get("N_temp", 1)))

                        # --- Wavelength SEM ---
                        w_mean = np.mean(wav_data)
                        w_sem = np.std(wav_data, ddof=1) / np.sqrt(len(wav_data))

                        t_means.append(t_mean)
                        t_sems.append(t_sem)
                        w_means.append(w_mean)
                        w_sems.append(w_sem)

        x = np.array(t_means)
        y = np.array(w_means)
        x_err = np.array(t_sems)
        y_err = np.array(w_sems)

        mask_valid = ~np.isnan(x) & ~np.isnan(y)
        x, y, x_err, y_err = x[mask_valid], y[mask_valid], x_err[mask_valid], y_err[mask_valid]

        if len(x) > 1:
            slope, intercept, r_val, p_val, std_err = stats.linregress(x, y)
            slope_pm, err_pm = slope * 1e3, std_err * 1e3
            
            summary_results.append({
                "Sensor": f"S{sensor_num}",
                "Type": config["title"],
                "Slope": slope_pm,
                "Error": err_pm,
                "R2": r_val**2
            })

            # --- PLOT DE ALTA VISIBILIDAD (SEM) ---
            ax.errorbar(
                x, y,
                xerr=x_err,
                yerr=y_err,
                fmt='o',
                color=config["color"],
                ecolor='black',
                elinewidth=2.5,
                capsize=6,
                markersize=3,   # reducido para visibilidad de barras
                label='Experimental Data'
            )

            # Fit line
            ax.plot(
                x,
                slope*x + intercept,
                color='black',
                linestyle='--',
                linewidth=3,
                alpha=0.7
            )

            # Legend with slope
            slope_label = f"SLOPE: {slope_pm:.2f} ± {err_pm:.2f} pm/K\n$R^2$: {r_val**2:.4f}"
            leg = ax.legend(
                [slope_label],
                loc='upper left',
                fontsize=18,
                frameon=True,
                shadow=True,
                facecolor='white',
                edgecolor='black'
            )
            plt.setp(leg.get_texts(), fontweight='black')

            # Titles and labels (NO mention of SEM here except global title)
            ax.set_title(f"SENSOR {sensor_num} | {config['title']}", fontsize=22, fontweight='black', pad=15)
            ax.set_xlabel("Temperature (K)", fontsize=18, fontweight='bold')
            ax.set_ylabel("Wavelength (nm)", fontsize=18, fontweight='bold')

            ax.tick_params(axis='both', which='major', labelsize=16)
            for label_tick in ax.get_xticklabels() + ax.get_yticklabels():
                label_tick.set_fontweight('bold')

            ax.yaxis.set_major_formatter(ticker.ScalarFormatter(useOffset=True))
            ax.yaxis.get_offset_text().set_fontsize(16)
            ax.yaxis.get_offset_text().set_fontweight('bold')

            ax.grid(True, alpha=0.4, linestyle='--', linewidth=1.5)

# Global title ONLY mentions SEM
plt.suptitle(
    f"SENSITIVITY ANALYSIS SUMMARY (SEM): {fiber_info}",
    fontsize=32,
    y=1.02,
    fontweight='black',
    color='darkblue'
)

plt.tight_layout()
plt.show()

# ============================================================================
# FINAL SUMMARY TABLE (IMPRESIÓN EN CONSOLA)
# ============================================================================
print("\n" + "="*95)
print(f"{'SENSITIVITY SUMMARY TABLE (SEM): FIBER DW-3 (Run 1)':^95}")
print("="*95)
print(f"{'Sensor':<8} | {'Type':<15} | {'Slope (pm/K)':<18} | {'Error (pm/K)':<15} | {'R²':<8}")
print("-" * 95)

for res in summary_results:
    print(f"{res['Sensor']:<8} | {res['Type']:<15} | {res['Slope']:>12.3f}       | {res['Error']:>12.3f}    | {res['R2']:.4f}")

print("="*95)

In [ ]:
# ============================================================================
# COMPARISON: STD vs SEM ERROR REPRESENTATION
# Fiber: DW-3 | Run 1 | Justification Figure
# ============================================================================

import matplotlib.pyplot as plt
from scipy import stats
import numpy as np
import matplotlib.ticker as ticker

plt.rcParams.update({'font.weight': 'bold', 'axes.labelweight': 'bold'})

fiber_info = "Fiber: DW-3 (Ormocer-T) | Run 1 | 2026-02-02"

fig, axes = plt.subplots(3, 2, figsize=(18, 20))  # col0=STD, col1=SEM

for s_idx, sensor_num in enumerate([1, 2, 3]):

    configs = [
        {"key": f"FBG-{sensor_num}-P", "color": "firebrick"},
        {"key": f"FBG-{sensor_num}-S", "color": "royalblue"},
        {"key": f"FBG-{sensor_num}-Mean", "color": "forestgreen"}
    ]

    for c_idx, cfg in enumerate(configs):

        label = cfg["key"]

        t_mean, t_std, t_sem = [], [], []
        w_mean, w_std, w_sem = [], [], []

        if 'plateau_data' in globals():

            for p in plateau_data:
                source = fbg_mean_data if "Mean" in label else fbg_filtered_data

                if label in source:
                    mask = (source[label]["timestamps"] >= p["t0"]) & \
                           (source[label]["timestamps"] <= p["tfin"])

                    wav = source[label]["wavelength"][mask]

                    if len(wav) > 1:

                        # --- X (Temperature) ---
                        Tm = p["temp_mean"]
                        Ts = p["temp_std"]
                        Tsem = Ts / np.sqrt(max(1, p.get("N_temp", len(wav))))

                        # --- Y (Wavelength) ---
                        Wm = np.mean(wav)
                        Ws = np.std(wav, ddof=1)
                        Wsem = Ws / np.sqrt(len(wav))

                        t_mean.append(Tm)
                        t_std.append(Ts)
                        t_sem.append(Tsem)

                        w_mean.append(Wm)
                        w_std.append(Ws)
                        w_sem.append(Wsem)

        x_mean = np.array(t_mean)
        y_mean = np.array(w_mean)

        # =========================
        # LEFT COLUMN → STD
        # =========================
        ax_std = axes[s_idx, 0]

        ax_std.errorbar(
            x_mean, y_mean,
            xerr=np.array(t_std),
            yerr=np.array(w_std),
            fmt='o',
            markersize=3,
            ecolor='black',
            elinewidth=2.5,
            capsize=6,
            color=cfg["color"]
        )

        if len(x_mean) > 1:
            slope, intercept, r, *_ = stats.linregress(x_mean, y_mean)
            ax_std.plot(x_mean, slope*x_mean + intercept, 'k--', alpha=0.6)

        ax_std.set_title(f"SENSOR {sensor_num} | STD", fontweight='bold')

        # =========================
        # RIGHT COLUMN → SEM
        # =========================
        ax_sem = axes[s_idx, 1]

        ax_sem.errorbar(
            x_mean, y_mean,
            xerr=np.array(t_sem),
            yerr=np.array(w_sem),
            fmt='o',
            markersize=3,
            ecolor='black',
            elinewidth=2.5,
            capsize=6,
            color=cfg["color"]
        )

        if len(x_mean) > 1:
            slope, intercept, r, *_ = stats.linregress(x_mean, y_mean)
            ax_sem.plot(x_mean, slope*x_mean + intercept, 'k--', alpha=0.6)

        ax_sem.set_title(f"SENSOR {sensor_num} | SEM", fontweight='bold')

        # --- Shared styling ---
        for ax in [ax_std, ax_sem]:
            ax.set_xlabel("Temperature (K)", fontweight='bold')
            ax.set_ylabel("Wavelength (nm)", fontweight='bold')
            ax.grid(True, alpha=0.3)
            ax.tick_params(labelsize=12)
            for t in ax.get_xticklabels() + ax.get_yticklabels():
                t.set_fontweight('bold')

fig.suptitle(
    f"STD vs SEM COMPARISON (Error Representation Justification)\n{fiber_info}",
    fontsize=20,
    fontweight='bold',
    y=1.02
)

plt.tight_layout()
plt.show()

## 7️⃣ Evolución Temporal Completa (Todo el rango con TIME_SHIFT aplicado)

In [ ]:
# ============================================================================
# CONFIGURACIÓN: Filtrar por rango temporal (opcional)
# ============================================================================
# Opción A: Mostrar TODO el rango temporal disponible
use_filtered_range_plot = True  # Cambiar a True para filtrar por un rango específico

if use_filtered_range_plot:
    # Opción B: Definir rango específico para visualización
    # Puedes usar el mismo rango de la Sección 3 o definir uno personalizado
    t_plot_inicio = datetime.datetime.combine(fecha, datetime.time(17, 12))
    t_plot_fin = datetime.datetime.combine(fecha, datetime.time(17, 15))
    range_label = f"({t_plot_inicio.strftime('%H:%M')}-{t_plot_fin.strftime('%H:%M')})"
else:
    # Mostrar todo el rango disponible
    t_plot_inicio = times_temp[0]
    t_plot_fin = times_temp[-1]
    range_label = "(Rango completo)"
# ============================================================================

fig, axes = plt.subplots(3, 1, figsize=(18, 12), sharex=True)

# Filtrar datos de temperatura según rango seleccionado
temp_raw_valid_mask = (temp_raw[:, rtd_ref_idx] > 0) & \
                       (times_temp >= t_plot_inicio) & \
                       (times_temp <= t_plot_fin)
times_temp_plot = times_temp[temp_raw_valid_mask]
temp_plot = temp_raw[temp_raw_valid_mask, rtd_ref_idx]

for idx, sensor_num in enumerate([1, 2, 3]):
    fbg_label = f"FBG-{sensor_num}-Mean"
    
    # Extraer datos RAW completos
    sensor_idx = sensor_num - 1  # 0-indexed
    wav_p_full = wav_raw[:, 0, sensor_idx]  # Polarización P
    wav_s_full = wav_raw[:, 1, sensor_idx]  # Polarización S
    
    # Filtrar valores positivos y por rango temporal
    mask_p = (wav_p_full > 0) & (times_peak >= t_plot_inicio) & (times_peak <= t_plot_fin)
    mask_s = (wav_s_full > 0) & (times_peak >= t_plot_inicio) & (times_peak <= t_plot_fin)
    
    times_p_plot = times_peak[mask_p]
    times_s_plot = times_peak[mask_s]
    wav_p_plot_valid = wav_p_full[mask_p]
    wav_s_plot_valid = wav_s_full[mask_s]
    
    # Interpolar S a tiempos de P
    times_p_sec = np.array([t.timestamp() for t in times_p_plot])
    times_s_sec = np.array([t.timestamp() for t in times_s_plot])
    wav_s_interp_plot = np.interp(times_p_sec, times_s_sec, wav_s_plot_valid)
    
    # Media (P+S)/2
    wav_mean_plot = (wav_p_plot_valid + wav_s_interp_plot) / 2.0
    
    ax = axes[idx]
    
    # Plot temperatura
    ax.plot(times_temp_plot, temp_plot, 'b-', linewidth=1.0, alpha=0.6, label=f'{rtd_ref_name}')
    ax.set_ylabel('Temperatura (K)', fontsize=10, color='b')
    ax.tick_params(axis='y', labelcolor='b')
    ax.grid(True, alpha=0.3)
    
    # Plot wavelength en eje secundario
    ax2 = ax.twinx()
    ax2.plot(times_p_plot, wav_mean_plot, 'r-', linewidth=0.8, alpha=0.7, label=f'FBG-{sensor_num} Mean')
    ax2.set_ylabel('Wavelength (nm)', fontsize=10, color='r')
    ax2.tick_params(axis='y', labelcolor='r')
    
    # Leyenda combinada
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=9)
    
    ax.set_title(f'FBG-{sensor_num}: Evolución Temporal {range_label} (TIME_SHIFT = +{TIME_SHIFT.total_seconds()/3600:.0f}h aplicado a wavelength)', 
                 fontsize=11)

# Configurar eje X solo en el último subplot
axes[-1].set_xlabel('Tiempo', fontsize=11)
for label in axes[-1].get_xticklabels():
    label.set_rotation(45)
    label.set_ha('right')

plt.suptitle(f'Evolución Temporal: {rtd_ref_name} vs FBG Sensors (con TIME_SHIFT)', 
             fontsize=13, y=0.995)
plt.tight_layout()
plt.show()

print(f"\n📊 Rango temporal mostrado: {range_label}")
print(f"   Temperatura: {times_temp_plot[0].strftime('%Y-%m-%d %H:%M')} - {times_temp_plot[-1].strftime('%H:%M')}")
print(f"   Wavelength (con +{TIME_SHIFT.total_seconds()/3600:.0f}h shift): {times_p_plot[0].strftime('%Y-%m-%d %H:%M')} - {times_p_plot[-1].strftime('%H:%M')}")
print(f"\n💡 Este gráfico muestra el TIME_SHIFT de {TIME_SHIFT.total_seconds()/3600:.0f} horas aplicado a los datos de wavelength")
print(f"   para alinearlos temporalmente con la temperatura.")
if use_filtered_range_plot:
    print(f"\n⚙️  Para ver el rango completo: cambia 'use_filtered_range_plot = False'")
else:
    print(f"\n⚙️  Para filtrar por rango temporal específico: cambia 'use_filtered_range_plot = True'")